# Molecular Activity Prediction
MLP + GCN (baseline) + GIN — scaffold split — MLflow tracking

In [2]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import mlflow
import mlflow.pytorch

from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool, global_add_pool
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR   = Path('../data')
print(f'Device: {DEVICE}')

c:\Users\natalia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 1. Scaffold split

In [10]:
def get_scaffold(smiles: str) -> str:
    """Return Bemis-Murcko scaffold SMILES, or the molecule itself if no scaffold."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return smiles
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold if scaffold else smiles
    except Exception:
        return smiles


def scaffold_split(
    smiles_list: list[str],
    val_frac:  float = 0.1,
    test_frac: float = 0.1,
    seed: int = SEED,
) -> tuple[list[int], list[int], list[int]]:
    """
    Bemis-Murcko scaffold split.
    Groups molecules by scaffold, then assigns whole scaffold groups to
    train/val/test — so no scaffold appears in more than one split.
    Returns index lists for train, val, test.
    """
    # Map each molecule index to its scaffold
    scaffold_to_indices: dict[str, list[int]] = {}
    for i, smi in enumerate(smiles_list):
        scaffold = get_scaffold(smi)
        scaffold_to_indices.setdefault(scaffold, []).append(i)

    # Sort scaffold groups by size descending — puts large families in train
    groups = sorted(scaffold_to_indices.values(), key=len, reverse=True)

    n          = len(smiles_list)
    val_size   = int(np.floor(val_frac  * n))
    test_size  = int(np.floor(test_frac * n))

    train_idx, val_idx, test_idx = [], [], []

    for group in groups:
        if len(test_idx) < test_size:
            test_idx.extend(group)
        elif len(val_idx) < val_size:
            val_idx.extend(group)
        else:
            train_idx.extend(group)

    print(f'Scaffold split → train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}')
    print(f'Unique scaffolds: {len(groups)}')
    return train_idx, val_idx, test_idx

## 2. MLP data

In [4]:
# Load latest prepared MLP parquet
mlp_files = sorted(DATA_DIR.glob('mlp_prepared_*.parquet'), reverse=True)
assert mlp_files, 'No mlp_prepared_*.parquet found in data/'
mlp_path  = mlp_files[0]
print(f'MLP source: {mlp_path.name}')

df_mlp = pd.read_parquet(mlp_path)
print(f'Rows: {len(df_mlp)}, columns: {list(df_mlp.columns)}')
df_mlp.head(2)

MLP source: mlp_prepared_2147_20260604_222802.parquet
Rows: 2520, columns: ['activity_id', 'pic50', 'features']


,activity_id,pic50,features
0,1723376,5.036212,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,1723466,5.036212,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [5]:
# Reconstruct SMILES for scaffold split — load from the raw joined file
raw_files = sorted(DATA_DIR.glob('chembl_joined_2147_*.parquet'), reverse=True)
assert raw_files, 'No chembl_joined_2147_*.parquet found in data/'
df_raw    = pd.read_parquet(raw_files[0], columns=['activity_id', 'canonical_smiles'])

df_mlp    = df_mlp.merge(df_raw, on='activity_id', how='left')

# Scaffold split
smiles_list                    = df_mlp['canonical_smiles'].tolist()
train_idx, val_idx, test_idx   = scaffold_split(smiles_list)

# Feature matrix
X_all = np.vstack(df_mlp['features'].values)       # (n, 2056)
y_all = df_mlp['pic50'].values.astype(np.float32)  # (n,)

# Scale physchem cols (last 8) on train only — Morgan bits [0:2048] stay as-is
scaler   = RobustScaler()
X_train  = X_all[train_idx].copy()
X_val    = X_all[val_idx].copy()
X_test   = X_all[test_idx].copy()

X_train[:, 2048:] = scaler.fit_transform(X_train[:, 2048:])
X_val[:,   2048:] = scaler.transform(X_val[:,   2048:])
X_test[:,  2048:] = scaler.transform(X_test[:,  2048:])

y_train = y_all[train_idx]
y_val   = y_all[val_idx]
y_test  = y_all[test_idx]

def to_tensor_dataset(X, y):
    return TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    )

MLP_BATCH = 64
mlp_train_loader = DataLoader(to_tensor_dataset(X_train, y_train), batch_size=MLP_BATCH, shuffle=True)
mlp_val_loader   = DataLoader(to_tensor_dataset(X_val,   y_val),   batch_size=MLP_BATCH)
mlp_test_loader  = DataLoader(to_tensor_dataset(X_test,  y_test),  batch_size=MLP_BATCH)

print(f'Feature dim: {X_train.shape[1]}  (2048 Morgan + 8 physchem)')

Scaffold split → train: 1978, val: 266, test: 276
Unique scaffolds: 1104
Feature dim: 2056  (2048 Morgan + 8 physchem)


## 3. GNN data

In [6]:
gnn_files = sorted(DATA_DIR.glob('gnn_graphs_*.pt'), reverse=True)
assert gnn_files, 'No gnn_graphs_*.pt found in data/'
gnn_path  = gnn_files[0]
print(f'GNN source: {gnn_path.name}')

all_graphs = torch.load(gnn_path)
print(f'Graphs: {len(all_graphs)}')
print(f'Node feature dim : {all_graphs[0].x.shape[1]}')
print(f'Edge feature dim : {all_graphs[0].edge_attr.shape[1]}')

NODE_DIM = all_graphs[0].x.shape[1]
EDGE_DIM = all_graphs[0].edge_attr.shape[1]

GNN source: gnn_graphs_20260604_223001_part0001.pt


C:\Users\natalia\AppData\Local\Temp\ipykernel_24344\1732275856.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  all_graphs = torch.load(gnn_path)


Graphs: 2520
Node feature dim : 28
Edge feature dim : 11


In [7]:
# Scaffold split for GNN — use stored smiles attribute
gnn_smiles                              = [g.smiles for g in all_graphs]
g_train_idx, g_val_idx, g_test_idx     = scaffold_split(gnn_smiles)

GNN_BATCH = 32
gnn_train_loader = GeoDataLoader([all_graphs[i] for i in g_train_idx], batch_size=GNN_BATCH, shuffle=True)
gnn_val_loader   = GeoDataLoader([all_graphs[i] for i in g_val_idx],   batch_size=GNN_BATCH)
gnn_test_loader  = GeoDataLoader([all_graphs[i] for i in g_test_idx],  batch_size=GNN_BATCH)

Scaffold split → train: 1978, val: 266, test: 276
Unique scaffolds: 1105


C:\Users\natalia\AppData\Local\Temp\ipykernel_24344\529663762.py:6: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  gnn_train_loader = GeoDataLoader([all_graphs[i] for i in g_train_idx], batch_size=GNN_BATCH, shuffle=True)
C:\Users\natalia\AppData\Local\Temp\ipykernel_24344\529663762.py:7: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  gnn_val_loader   = GeoDataLoader([all_graphs[i] for i in g_val_idx],   batch_size=GNN_BATCH)
C:\Users\natalia\AppData\Local\Temp\ipykernel_24344\529663762.py:8: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  gnn_test_loader  = GeoDataLoader([all_graphs[i] for i in g_test_idx],  batch_size=GNN_BATCH)


## 4. Model definitions

In [8]:
# ── MLP ───────────────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float = 0.3):
        super().__init__()
        dims   = [input_dim] + hidden_dims + [1]
        layers = []
        for i in range(len(dims) - 2):
            layers += [
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
        layers.append(nn.Linear(dims[-2], dims[-1]))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ── GCN (baseline) ────────────────────────────────────────────────────────────
class GCN(nn.Module):
    def __init__(self, node_dim: int, hidden_dim: int, num_layers: int, dropout: float = 0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        dims = [node_dim] + [hidden_dim] * num_layers
        for i in range(num_layers):
            self.convs.append(GCNConv(dims[i], dims[i+1]))
            self.bns.append(nn.BatchNorm1d(dims[i+1]))

        self.dropout = dropout
        self.head    = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.head(x)


# ── GIN (main GNN) ────────────────────────────────────────────────────────────
class GIN(nn.Module):
    def __init__(self, node_dim: int, hidden_dim: int, num_layers: int, dropout: float = 0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        dims = [node_dim] + [hidden_dim] * num_layers
        for i in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(dims[i],    hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        self.dropout = dropout
        # Sum pooling recommended for GIN (original paper)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_add_pool(x, batch)   # sum pooling for GIN
        return self.head(x)

## 5. Training utilities

In [11]:
def train_epoch_mlp(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def eval_mlp(model, loader, criterion):
    model.eval()
    total_loss, preds, targets = 0, [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            out  = model(X_batch)
            total_loss += criterion(out, y_batch).item() * len(X_batch)
            preds.append(out.cpu());  targets.append(y_batch.cpu())
    preds   = torch.cat(preds).numpy().squeeze()
    targets = torch.cat(targets).numpy().squeeze()
    return total_loss / len(loader.dataset), r2_score(targets, preds), mean_absolute_error(targets, preds)


def train_epoch_gnn(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(batch), batch.y.unsqueeze(1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


def eval_gnn(model, loader, criterion):
    model.eval()
    total_loss, preds, targets = 0, [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            out   = model(batch)
            total_loss += criterion(out, batch.y.unsqueeze(1)).item() * batch.num_graphs
            preds.append(out.cpu());  targets.append(batch.y.cpu())
    preds   = torch.cat(preds).numpy().squeeze()
    targets = torch.cat(targets).numpy().squeeze()
    return total_loss / len(loader.dataset), r2_score(targets, preds), mean_absolute_error(targets, preds)


def train_model(model, train_loader, val_loader, epochs, lr, model_type='gnn', patience=20):
    """Generic training loop — works for MLP and GNN via model_type switch."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    criterion = nn.MSELoss()

    train_fn = train_epoch_mlp if model_type == 'mlp' else train_epoch_gnn
    eval_fn  = eval_mlp        if model_type == 'mlp' else eval_gnn

    best_val_loss  = float('inf')
    best_state     = None
    patience_count = 0

    for epoch in range(1, epochs + 1):
        train_loss                    = train_fn(model, train_loader, optimizer, criterion)
        val_loss, val_r2, val_mae     = eval_fn(model,  val_loader,   criterion)
        scheduler.step(val_loss)

        mlflow.log_metrics({
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'val_r2':     val_r2,
            'val_mae':    val_mae,
        }, step=epoch)

        if epoch % 10 == 0:
            print(f'Epoch {epoch:>3} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_R²: {val_r2:.3f} | val_MAE: {val_mae:.3f}')

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return model

## 6. Train MLP

In [12]:
mlflow.set_experiment('chembl_2147_activity_prediction')

MLP_PARAMS = {
    'hidden_dims': [512, 256, 128],
    'dropout':     0.3,
    'lr':          1e-3,
    'epochs':      150,
    'batch_size':  MLP_BATCH,
}

with mlflow.start_run(run_name='MLP'):
    mlflow.log_params(MLP_PARAMS)
    mlflow.log_param('split', 'scaffold_bemis_murcko')
    mlflow.log_param('input_dim', X_train.shape[1])
    mlflow.log_param('train_size', len(train_idx))
    mlflow.log_param('val_size',   len(val_idx))
    mlflow.log_param('test_size',  len(test_idx))

    model_mlp = MLP(
        input_dim   = X_train.shape[1],
        hidden_dims = MLP_PARAMS['hidden_dims'],
        dropout     = MLP_PARAMS['dropout'],
    ).to(DEVICE)

    model_mlp = train_model(
        model_mlp, mlp_train_loader, mlp_val_loader,
        epochs=MLP_PARAMS['epochs'], lr=MLP_PARAMS['lr'], model_type='mlp'
    )

    criterion = nn.MSELoss()
    test_loss, test_r2, test_mae = eval_mlp(model_mlp, mlp_test_loader, criterion)
    mlflow.log_metrics({'test_loss': test_loss, 'test_r2': test_r2, 'test_mae': test_mae})
    mlflow.pytorch.log_model(model_mlp, 'model')

    print(f'\nMLP test → RMSE: {test_loss**0.5:.4f} | R²: {test_r2:.3f} | MAE: {test_mae:.3f}')

2026/06/04 22:57:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/04 22:57:07 INFO mlflow.store.db.utils: Updating database tables
2026/06/04 22:57:09 INFO mlflow.tracking.fluent: Experiment with name 'chembl_2147_activity_prediction' does not exist. Creating a new experiment.


Epoch  10 | train_loss: 0.9697 | val_loss: 1.2714 | val_R²: 0.417 | val_MAE: 0.806
Epoch  20 | train_loss: 0.8536 | val_loss: 1.2364 | val_R²: 0.433 | val_MAE: 0.860
Epoch  30 | train_loss: 0.7170 | val_loss: 0.9501 | val_R²: 0.565 | val_MAE: 0.693


2026/06/04 22:57:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 22:57:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/04 22:57:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Early stopping at epoch 37


2026/06/04 22:57:21 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.



MLP test → RMSE: 1.5950 | R²: -0.235 | MAE: 1.016


## 7. Train GCN (baseline)

In [13]:
GCN_PARAMS = {
    'hidden_dim': 128,
    'num_layers': 3,
    'dropout':    0.3,
    'lr':         1e-3,
    'epochs':     150,
    'batch_size': GNN_BATCH,
}

with mlflow.start_run(run_name='GCN_baseline'):
    mlflow.log_params(GCN_PARAMS)
    mlflow.log_param('split',     'scaffold_bemis_murcko')
    mlflow.log_param('node_dim',  NODE_DIM)
    mlflow.log_param('edge_dim',  EDGE_DIM)
    mlflow.log_param('train_size', len(g_train_idx))
    mlflow.log_param('val_size',   len(g_val_idx))
    mlflow.log_param('test_size',  len(g_test_idx))

    model_gcn = GCN(
        node_dim   = NODE_DIM,
        hidden_dim = GCN_PARAMS['hidden_dim'],
        num_layers = GCN_PARAMS['num_layers'],
        dropout    = GCN_PARAMS['dropout'],
    ).to(DEVICE)

    model_gcn = train_model(
        model_gcn, gnn_train_loader, gnn_val_loader,
        epochs=GCN_PARAMS['epochs'], lr=GCN_PARAMS['lr'], model_type='gnn'
    )

    criterion = nn.MSELoss()
    test_loss, test_r2, test_mae = eval_gnn(model_gcn, gnn_test_loader, criterion)
    mlflow.log_metrics({'test_loss': test_loss, 'test_r2': test_r2, 'test_mae': test_mae})
    mlflow.pytorch.log_model(model_gcn, 'model')

    print(f'\nGCN test → RMSE: {test_loss**0.5:.4f} | R²: {test_r2:.3f} | MAE: {test_mae:.3f}')

Epoch  10 | train_loss: 2.4507 | val_loss: 1.4698 | val_R²: 0.326 | val_MAE: 0.958
Epoch  20 | train_loss: 2.1243 | val_loss: 1.8185 | val_R²: 0.167 | val_MAE: 1.153
Epoch  30 | train_loss: 2.0517 | val_loss: 2.5254 | val_R²: -0.157 | val_MAE: 1.383
Epoch  40 | train_loss: 2.0053 | val_loss: 1.3209 | val_R²: 0.395 | val_MAE: 0.955


2026/06/04 23:02:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 23:02:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/04 23:02:43 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Early stopping at epoch 48


2026/06/04 23:02:49 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.



GCN test → RMSE: 1.3077 | R²: 0.170 | MAE: 0.908


## 8. Train GIN

In [ ]:
GIN_PARAMS = {
    'hidden_dim': 128,
    'num_layers': 4,
    'dropout':    0.3,
    'lr':         1e-3,
    'epochs':     150,
    'batch_size': GNN_BATCH,
}

with mlflow.start_run(run_name='GIN'):
    mlflow.log_params(GIN_PARAMS)
    mlflow.log_param('split',     'scaffold_bemis_murcko')
    mlflow.log_param('pooling',   'global_add_pool')
    mlflow.log_param('node_dim',  NODE_DIM)
    mlflow.log_param('edge_dim',  EDGE_DIM)
    mlflow.log_param('train_size', len(g_train_idx))
    mlflow.log_param('val_size',   len(g_val_idx))
    mlflow.log_param('test_size',  len(g_test_idx))

    model_gin = GIN(
        node_dim   = NODE_DIM,
        hidden_dim = GIN_PARAMS['hidden_dim'],
        num_layers = GIN_PARAMS['num_layers'],
        dropout    = GIN_PARAMS['dropout'],
    ).to(DEVICE)

    model_gin = train_model(
        model_gin, gnn_train_loader, gnn_val_loader,
        epochs=GIN_PARAMS['epochs'], lr=GIN_PARAMS['lr'], model_type='gnn'
    )

    criterion = nn.MSELoss()
    test_loss, test_r2, test_mae = eval_gnn(model_gin, gnn_test_loader, criterion)
    mlflow.log_metrics({'test_loss': test_loss, 'test_r2': test_r2, 'test_mae': test_mae})
    mlflow.pytorch.log_model(model_gin, 'model')

    print(f'\nGIN test → RMSE: {test_loss**0.5:.4f} | R²: {test_r2:.3f} | MAE: {test_mae:.3f}')

Epoch  10 | train_loss: 3.2942 | val_loss: 1.4725 | val_R²: 0.325 | val_MAE: 0.999
Epoch  20 | train_loss: 2.4991 | val_loss: 0.9765 | val_R²: 0.553 | val_MAE: 0.760
Epoch  30 | train_loss: 2.4211 | val_loss: 2.1859 | val_R²: -0.002 | val_MAE: 1.147
Epoch  40 | train_loss: 1.9260 | val_loss: 1.0704 | val_R²: 0.509 | val_MAE: 0.738
Epoch  50 | train_loss: 1.7843 | val_loss: 1.1695 | val_R²: 0.464 | val_MAE: 0.786


2026/06/04 23:03:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 23:03:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/04 23:03:42 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Early stopping at epoch 52


2026/06/04 23:03:48 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.



GIN test → RMSE: 1.8916 | R²: -0.737 | MAE: 1.155


: 

## 9. Compare results

In [ ]:
import mlflow

runs = mlflow.search_runs(
    experiment_names=['chembl_2147_activity_prediction'],
    order_by=['metrics.test_r2 DESC']
)

cols = ['tags.mlflow.runName', 'metrics.test_r2', 'metrics.test_mae', 'metrics.test_loss']
print(runs[cols].rename(columns={
    'tags.mlflow.runName': 'model',
    'metrics.test_r2':     'test_R²',
    'metrics.test_mae':    'test_MAE',
    'metrics.test_loss':   'test_MSE',
}).to_string(index=False))

print('\nRun `mlflow ui` in the project root to explore full training curves.')